# Análise de E-Commerce Brasileiro (Dataset Olist)
### Engenharia e Análise de Dados com DuckDB e SQL

Este projeto realiza a ingestão, modelagem analítica e extração de insights sobre pedidos, prazos de entrega, categorias mais vendidas e faturamento do marketplace Olist.

In [1]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "olist.duckdb"

# Conecta via DuckDB
con = duckdb.connect(str(db_path), read_only=True)

# Passa a conexão existente para a extensão SQL
%reload_ext sql
%sql con
%config SqlMagic.autolimit = 50

# Para exibir, por exemplo, até 50 linhas na tabela:
%config SqlMagic.displaylimit = 15

## 1. Visão Geral das Tabelas e Contagem de Registros
Verificamos a integridade das tabelas criadas após o processo de ingestão.

In [2]:
query_contagem = """
SELECT 'customers' AS tabela, count(*) AS total_registros FROM customers
UNION ALL
SELECT 'orders', count(*) FROM orders
UNION ALL
SELECT 'order_items', count(*) FROM order_items
UNION ALL
SELECT 'order_payments', count(*) FROM order_payments
UNION ALL
SELECT 'order_reviews', count(*) FROM order_reviews
UNION ALL
SELECT 'products', count(*) FROM products
UNION ALL
SELECT 'sellers', count(*) FROM sellers;
"""
con.execute(query_contagem).df()

,tabela,total_registros
0,customers,99441
1,orders,99441
2,order_items,112650
3,order_payments,103886
4,order_reviews,99224
5,products,32951
6,sellers,3095


## Olist Brazilian E-Commerce: "Why are customers leaving bad reviews?

Our teams noticed that customer satisfaction has been decreasing and our team thinks it was caused due to the late deliveries. to answer that we need to do a set of queries to find out.

### What percentage of orders are delivered later than their estimated delivery date?

In [3]:
%%sql
--to answer this question we´ll be using the CASE.
--the CASE function allows us to filter when a delivery was a delay or not using date arithmetic
WITH orders_relation AS (
    SELECT
        CASE
            WHEN DATE_PART('day', order_estimated_delivery_date - order_delivered_customer_date) < 0 THEN 'atrasado'
            ELSE 'no_prazo_ou_adiantado'
        END AS status_entrega
    FROM orders
    WHERE order_status = 'delivered' 
      AND order_delivered_customer_date IS NOT NULL 
      AND order_estimated_delivery_date IS NOT NULL
)

/************************************************************************************************
after that we need to sum all the delayed orders.
we need to sum where the status_entrea = delayed/not_delayed, if it´s THEN 1 if not THEN 0
but when we devide an integer by another integer sql omites the decimal value letting a inprecise
number, so to solve that we use the cast operator ::decimal it converts the number on its left
turning it into a decimal value.
*************************************************************************************************/

SELECT
    (SUM(CASE WHEN status_entrega = 'atrasado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_atrasados,
    (SUM(CASE WHEN status_entrega = 'no_prazo_ou_adiantado' THEN 1 ELSE 0 END)::decimal / COUNT(*)) * 100 AS pct_entregues_no_prazo
FROM orders_relation;


Running query in 'DuckDBPyConnection'

pct_atrasados,pct_entregues_no_prazo
6.7730900798175595,93.22690992018245


> **Conclusão:** 
> Apenas **6,77%** das encomendas foram entregues com atraso. Isto indica que as encomendas atrasadas representam um grupo reduzido e que outros fatores podem estar a influenciar a insatisfação geral dos clientes.

## Is there a relationship between delivery delay and review score? (e.g., average review score for on-time vs. late orders)

In [4]:
%%sql
WITH delay_infor AS (
    SELECT
        CASE
            WHEN DATE_PART('day', order_estimated_delivery_date - order_delivered_customer_date) < 0 THEN 'delayed'
            ELSE 'delivered_on_time'
        END AS delivery_status,
        review_score
    FROM orders o 
    JOIN order_reviews os
        ON o.order_id = os.order_id
    WHERE order_status == 'delivered'
        AND order_estimated_delivery_date IS NOT NULL
        AND order_delivered_customer_date IS NOT NULL
)

SELECT
    delivery_status,
    AVG(review_score)
FROM delay_infor
GROUP BY delivery_status;


Running query in 'DuckDBPyConnection'

delivery_status,avg(review_score)
delivered_on_time,4.289980432268967
delayed,2.271025120923701


> **Conclusion**
>The analysis confirms the team’s hypothesis: delivery delays are directly linked to falling satisfaction. Customers who had received on delay has given reviews on average of 2.27 while customers that received on time has given an avg of 4.28.

>that means that delays has a significant impact on customers satisfaction almost cutting the score by half. With bigger delays the less will be the rating score.

>If the marketplace dont enhance the logistics, the reputation and customers retention could get in risk

## Which product categories have the worst average review scores? Which have the best?

In [10]:
%%sql
SELECT
    product_category_name,
    AVG(review_score)
FROM products p
JOIN order_items oi
    ON p.product_id = oi.product_id
JOIN order_reviews oe
    ON oe.order_id = oi.order_id
GROUP BY product_category_name
ORDER BY AVG(review_score) DESC;

Running query in 'DuckDBPyConnection'

product_category_name,avg(review_score)
cds_dvds_musicais,4.642857142857143
fashion_roupa_infanto_juvenil,4.5
livros_interesse_geral,4.446265938069216
construcao_ferramentas_ferramentas,4.444444444444445
flores,4.419354838709677
livros_importados,4.4
livros_tecnicos,4.368421052631579
alimentos_bebidas,4.315412186379929
malas_acessorios,4.315257352941177
portateis_casa_forno_e_cafe,4.302631578947368


> **Conclusion:**
> Our initial hypothesis was that declining customer satisfaction was primarily driven by delivery delays. While the data confirms that late deliveries have a significant negative impact on review scores, it may not be the only factor. Product quality and customer expectations also appear to play a role. The query above highlights the top 10 product categories ranked by average review score, showing that some categories consistently receive higher ratings (such as children’s clothing and CDs/DVDs), while others score lower. This suggests that both logistics and product-related factors influence overall customer satisfaction.


## **Which sellers have the highest rate of late deliveries? Should any be flagged for a performance review?**

In [ ]:
%%sql
WITH delays_info AS (
    SELECT
        CASE
            WHEN (order_delivered_customer_date > order_estimated_delivery_date) THEN 'delayed'
            ELSE 'not_delayed'
        END AS delays_status,
        seller_id
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE order_status = 'delivered'
        AND order_delivered_customer_date IS NOT NULL
        AND order_estimated_delivery_date IS NOT NULL
),

delays_per_seller AS (
    SELECT
        seller_id,
        SUM(CASE WHEN delays_status = 'delayed' THEN 1 ELSE 0 END) delayed_orders
    FROM delays_info
    GROUP BY seller_id
),

total_delays AS (
    SELECT
        SUM(CASE WHEN delays_status = 'delayed') as total_delayed_orders
    FROM delays_per_seller
)

SELECT
    d.seller_id AS seller,
    d.delayed_orders,
    ROUND(d.delayed_orders::decimal / t.total_delayed_orders * 100, 2) || '%' AS percent_of_total_delays
FROM delays_per_seller d
CROSS JOIN total_delays t
ORDER BY d.delayed_orders DESC;


Running query in 'DuckDBPyConnection'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:
Parser Error: syntax error at or near ")"

LINE 26:         SUM(CASE WHEN delays_status = 'delayed') as total_delayed_orders
                                                        ^



## What is the average time between order purchase and order delivery, broken down by state/region?